<a href="https://colab.research.google.com/github/Nithiiii-07/ApacheSpark/blob/main/Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pyspark.sql import SparkSession

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
spark=SparkSession.builder.appName("Nithish").getOrCreate()

In [7]:
data=spark.read.csv("/content/drive/MyDrive/customer_churn.csv",inferSchema=True,header=True)

In [10]:
data.show()
data.printSchema()

+-------------------+----+--------------+---------------+-----+---------+-------------------+--------------------+--------------------+-----+
|              Names| Age|Total_Purchase|Account_Manager|Years|Num_Sites|       Onboard_date|            Location|             Company|Churn|
+-------------------+----+--------------+---------------+-----+---------+-------------------+--------------------+--------------------+-----+
|   Cameron Williams|42.0|       11066.8|              0| 7.22|      8.0|2013-08-30 07:00:40|10265 Elizabeth M...|          Harvey LLC|    1|
|      Kevin Mueller|41.0|      11916.22|              0|  6.5|     11.0|2013-08-13 00:38:46|6157 Frank Garden...|          Wilson PLC|    1|
|        Eric Lozano|38.0|      12884.75|              0| 6.67|     12.0|2016-06-29 06:20:07|1331 Keith Court ...|Miller, Johnson a...|    1|
|      Phillip White|42.0|       8010.76|              0| 6.71|     10.0|2014-04-22 12:43:12|13120 Daniel Moun...|           Smith Inc|    1|
|     

In [12]:
data_col=data.select(['Age','Total_Purchase','Account_Manager','Years', 'Num_Sites','Location','Churn'])

In [ ]:
data_col = data_col.na.drop(subset=['Location'])

In [13]:
data_col.show()

+----+--------------+---------------+-----+---------+--------------------+-----+
| Age|Total_Purchase|Account_Manager|Years|Num_Sites|            Location|Churn|
+----+--------------+---------------+-----+---------+--------------------+-----+
|42.0|       11066.8|              0| 7.22|      8.0|10265 Elizabeth M...|    1|
|41.0|      11916.22|              0|  6.5|     11.0|6157 Frank Garden...|    1|
|38.0|      12884.75|              0| 6.67|     12.0|1331 Keith Court ...|    1|
|42.0|       8010.76|              0| 6.71|     10.0|13120 Daniel Moun...|    1|
|37.0|       9191.58|              0| 5.56|      9.0|765 Tricia Row Ka...|    1|
|48.0|      10356.02|              0| 5.12|      8.0|6187 Olson Mounta...|    1|
|44.0|      11331.58|              1| 5.23|     11.0|4846 Savannah Roa...|    1|
|32.0|       9885.12|              1| 6.92|      9.0|25271 Roy Express...|    1|
|43.0|       14062.6|              1| 5.46|     11.0|3725 Caroline Str...|    1|
|40.0|       8066.94|       

In [14]:
from pyspark.ml.feature import (StringIndexer,VectorAssembler,OneHotEncoder)

In [35]:
Loc_indx=StringIndexer(inputCol='Location',outputCol='Loc_indx', handleInvalid='keep')

In [36]:
Loc_Vec=OneHotEncoder(inputCol='Loc_indx',outputCol='Loc_Vec')

In [37]:
assembler=VectorAssembler(inputCols=[
    'Age','Total_Purchase','Account_Manager','Years','Num_sites','Loc_Vec','Churn'
], outputCol='features')

In [38]:
from pyspark.ml import Pipeline

In [39]:
from pyspark.ml.classification import LogisticRegression

In [40]:
lr=LogisticRegression(featuresCol='features',labelCol='Churn')

In [41]:
pipeline= Pipeline(stages=[Loc_indx,Loc_Vec,assembler,lr])

In [42]:
train_data,test_data=data_col.randomSplit([0.7,0.3])

In [43]:
model=pipeline.fit(train_data)

In [44]:
final=model.transform(test_data)

In [45]:
final.select('Churn','prediction').show()

+-----+----------+
|Churn|prediction|
+-----+----------+
|    1|       1.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    1|       1.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    1|       1.0|
|    1|       1.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
+-----+----------+
only showing top 20 rows
